In [0]:
# ============================================================
# Notebook: 01_eda_profiling/01_eda_profiling_all_tables
# Project : COVID-19 Clinical Data Lakehouse
# Purpose : Comprehensive EDA & Data Profiling for all 16 tables in the bronze schema
#           prior to designing the Conceptual/Physical Model and DQ rules in Silver.
#
# All columns in Bronze are stored as String (Raw Truth). Profiling here
# focuses on: inferring logical schema, null/empty ratios, distinct values count,
# and data sampling — enabling informed decisions on target data types and DQ rules.
# ============================================================

from pyspark.sql import functions as F

catalog_name = "covid19_lakehouse"
schema_name = "bronze"

table_names = [
    "patients", "encounters", "observations", "conditions",
    "medications", "procedures", "careplans", "allergies",
    "devices", "imaging_studies", "immunizations", "organizations",
    "payers", "payer_transitions", "providers", "supplies",
]

# ============================================================
# Generic Profiling Function - Applicable to any table
# ============================================================
def profile_table(table_name: str):
    full_table_name = f"{catalog_name}.{schema_name}.{table_name}"
    df = spark.table(full_table_name)

    row_count = df.count()
    columns = [c for c in df.columns if not c.startswith("_")]  # Exclude metadata columns

    print("=" * 80)
    print(f"📊 TABLE: {table_name}   |   Rows: {row_count:,}   |   Columns: {len(columns)}")
    print("=" * 80)

    if row_count == 0:
        print("⚠️  Table is empty - verify the Ingestion step!\n")
        return {
            "table_name": table_name,
            "row_count": 0,
            "column_count": len(columns),
        }

    # --- 1) Null / Empty-string percentage per column ---
    null_exprs = [
        F.round(
            (F.sum(F.when((F.col(c).isNull()) | (F.trim(F.col(c)) == ""), 1).otherwise(0)) / row_count) * 100, 2
        ).alias(c)
        for c in columns
    ]
    null_pct_row = df.select(null_exprs).collect()[0].asDict()

    print("🔍 Null / Empty %:")
    for col_name, pct in sorted(null_pct_row.items(), key=lambda x: -x[1]):
        flag = "  ⚠️" if pct > 30 else ""
        print(f"   {col_name:<25} {pct:>6.2f}%{flag}")

    # --- 2) Distinct count per column (useful for identifying categorical values / primary keys) ---
    print("\n🔢 Distinct Values (approx):")
    distinct_exprs = [F.approx_count_distinct(F.col(c)).alias(c) for c in columns]
    distinct_row = df.select(distinct_exprs).collect()[0].asDict()
    for col_name, distinct_count in distinct_row.items():
        ratio = distinct_count / row_count if row_count > 0 else 0
        tag = "  (likely ID/PK)" if ratio > 0.95 else ("  (categorical)" if distinct_count <= 20 else "")
        print(f"   {col_name:<25} {distinct_count:>10,}{tag}")

    # --- 3) Data Sampling ---
    print("\n📋 Sample (3 rows):")
    df.select(columns).show(3, truncate=40, vertical=True)

    print("\n")

    return {
        "table_name": table_name,
        "row_count": row_count,
        "column_count": len(columns),
        "null_pct": null_pct_row,
        "distinct_count": distinct_row,
    }


# ============================================================
# Execute Profiling across all 16 tables + Aggregate Summary
# ============================================================
profiling_results = []

for table_name in table_names:
    result = profile_table(table_name)
    profiling_results.append(result)

# ============================================================
# Overall Final Summary for all Bronze Tables (Quick view of total scale)
# ============================================================
print("=" * 80)
print("OVERALL SUMMARY - ALL BRONZE TABLES")
print("=" * 80)
print(f"{'Table':<20} {'Rows':>15} {'Columns':>10}")
print("-" * 80)
total_rows = 0
for r in profiling_results:
    print(f"{r['table_name']:<20} {r['row_count']:>15,} {r['column_count']:>10}")
    total_rows += r["row_count"]
print("-" * 80)
print(f"{'TOTAL':<20} {total_rows:>15,}")

In [0]:
# ============================================================
# Notebook: 01_eda_profiling/02_deep_dive_profiling
# Purpose : Address critical gaps identified in initial profiling:
#           1) Date ranges per table (to design dim_date)
#           2) Referential integrity check between Fact tables and dim_patient
#           3) Exact Primary Key duplicate check (exact count, non-approximate)
#           4) Consistency check between observations.VALUE and observations.TYPE
#           5) Logical temporal consistency check for START/STOP dates (STOP < START)
# ============================================================

from pyspark.sql import functions as F

catalog_name = "covid19_lakehouse"
schema_name = "bronze"

# ============================================================
# 1) Date Range per Table (for tables with known date columns)
# ============================================================
date_columns_map = {
    "patients": ["BIRTHDATE", "DEATHDATE"],
    "encounters": ["START", "STOP"],
    "observations": ["DATE"],
    "conditions": ["START", "STOP"],
    "medications": ["START", "STOP"],
    "procedures": ["DATE"],
    "careplans": ["START", "STOP"],
    "allergies": ["START", "STOP"],
    "devices": ["START", "STOP"],
    "imaging_studies": ["DATE"],
    "immunizations": ["DATE"],
    "payer_transitions": ["START_YEAR", "END_YEAR"],
    "supplies": ["DATE"],
}

print("=" * 80)
print("1) DATE RANGE PER TABLE")
print("=" * 80)
for table_name, cols in date_columns_map.items():
    df = spark.table(f"{catalog_name}.{schema_name}.{table_name}")
    exprs = []
    for c in cols:
        exprs.append(F.min(F.col(c)).alias(f"{c}_min"))
        exprs.append(F.max(F.col(c)).alias(f"{c}_max"))
    row = df.select(exprs).collect()[0].asDict()
    print(f"\n📅 {table_name}:")
    for c in cols:
        print(f"   {c:<12} min={row[f'{c}_min']}   max={row[f'{c}_max']}")


# ============================================================
# 2) Referential Integrity Check - Is every PATIENT present in dim_patient?
# ============================================================
print("\n" + "=" * 80)
print("2) REFERENTIAL INTEGRITY CHECK (orphan PATIENT records)")
print("=" * 80)

patient_ids = spark.table(f"{catalog_name}.{schema_name}.patients").select("Id")

tables_with_patient_fk = [
    "encounters", "observations", "conditions", "medications",
    "procedures", "careplans", "allergies", "devices",
    "imaging_studies", "immunizations", "supplies", "payer_transitions",
]

for table_name in tables_with_patient_fk:
    df = spark.table(f"{catalog_name}.{schema_name}.{table_name}")
    total = df.count()
    orphans = df.join(patient_ids, df.PATIENT == patient_ids.Id, "left_anti").count()
    pct = round((orphans / total) * 100, 4) if total > 0 else 0
    flag = "  ⚠️" if orphans > 0 else "  ✅"
    print(f"{table_name:<20} Orphan rows: {orphans:>10,} / {total:,}  ({pct}%){flag}")


# ============================================================
# 3) Exact Primary Key Duplicate Check for tables with explicit Id
# ============================================================
print("\n" + "=" * 80)
print("3) EXACT PRIMARY KEY DUPLICATE CHECK")
print("=" * 80)

pk_candidates = {
    "patients": "Id",
    "encounters": "Id",
    "careplans": "Id",
    "imaging_studies": "Id",
    "organizations": "Id",
    "providers": "Id",
    "payers": "Id",
}

for table_name, pk_col in pk_candidates.items():
    df = spark.table(f"{catalog_name}.{schema_name}.{table_name}")
    total = df.count()
    exact_distinct = df.select(pk_col).distinct().count()
    duplicates = total - exact_distinct
    flag = "  ⚠️ DUPLICATES FOUND" if duplicates > 0 else "  ✅ unique"
    print(f"{table_name:<20} Rows: {total:>10,}   Distinct {pk_col}: {exact_distinct:>10,}   Dupes: {duplicates:>6,}{flag}")


# ============================================================
# 4) Consistency Check: observations.VALUE vs observations.TYPE
# ============================================================
print("\n" + "=" * 80)
print("4) OBSERVATIONS: VALUE vs TYPE consistency")
print("=" * 80)

obs = spark.table(f"{catalog_name}.{schema_name}.observations")

type_counts = obs.groupBy("TYPE").count().collect()
print("Distribution of TYPE column:")
for r in type_counts:
    print(f"   {r['TYPE']:<15} {r['count']:>12,}")

# Count rows where TYPE='numeric' but VALUE cannot be cast to double
non_numeric_but_marked_numeric = (
    obs.filter(F.col("TYPE") == "numeric")
    .filter(F.col("VALUE").cast("double").isNull() & F.col("VALUE").isNotNull())
    .count()
)
print(f"\n⚠️  Rows marked TYPE='numeric' but VALUE not castable to double: {non_numeric_but_marked_numeric:,}")


# ============================================================
# 5) Temporal Consistency Check: START vs STOP (Is STOP < START?)
# ============================================================
print("\n" + "=" * 80)
print("5) START/STOP LOGIC CHECK (STOP < START)")
print("=" * 80)

tables_with_start_stop = ["encounters", "conditions", "medications", "careplans", "devices", "allergies"]

for table_name in tables_with_start_stop:
    df = spark.table(f"{catalog_name}.{schema_name}.{table_name}")
    invalid = df.filter(
        F.col("STOP").isNotNull() & (F.col("STOP") < F.col("START"))
    ).count()
    flag = "  ⚠️️" if invalid > 0 else "  ✅"
    print(f"{table_name:<15} Invalid (STOP < START) rows: {invalid:,}{flag}")